# CS-781 Phase 7 — Experiment 6: Model 1 → Model 3 Knowledge Distillation

**Teacher:** Model 1 full-finetuned EMA checkpoint (Experiment 5; reported Test score 0.3314).  
**Student:** original Phase-1 Model 3 encoder–decoder, using the successful Model 3 encoder cross-encoder scoring formulation (benchmark 0.3473).

The teacher and student have different hidden sizes and architectures. Distillation is therefore performed on **query-group relevance logits**, not hidden states. The teacher is frozen. The student starts from its Phase-1 checkpoint.

**Data discipline:** teacher targets are generated only for Train query/candidate pairs. Dev is used only for checkpoint/fusion selection. Test labels are never accessed.

**Outputs:** all artifacts are written directly to `/kaggle/working/phase7_model1_to_model3_kd/` (flat; no subfolders).

## Changes in this improved version (core KD design unchanged)

Same teacher (Model 1 EMA), same student (Model 3, last 3 encoder blocks + scorer), same logit-level KD loss (KL + CE + pairwise), same BM25 -> rerank -> fusion pipeline and the same Train/Dev/Test discipline.

**Bug fixes**
- `StudentPairScorer` was used but never defined (NameError) -> defined (encoder, masked mean-pool, MLP head, mirrors the teacher scorer).
- `heapq` / `grad_checkpoint` imports added.
- The scorer wrapper contains the student backbone. The earlier `for p in student_head.parameters(): p.requires_grad=True` silently unfroze the entire backbone. This version only enables `student_head.score` and asserts that the first 6 encoder blocks and all decoder blocks remain frozen.
- Optimizer uses the last 3 encoder blocks + encoder LayerNorm + scoring MLP exactly once; separate parameter groups prevent duplicate backbone updates.
- Teacher-target cache now has a checkpoint/data/config signature, so stale cached logits are not reused after a change.
- Final partial gradient-accumulation windows are rescaled correctly instead of being underweighted.

**Score-oriented changes**
- Teacher targets: 3 different (positive, negatives) groups per query, 7 negatives per group (3 of them from the 15 hardest BM25 negatives); a new group each epoch instead of one frozen sample.
- Gradient accumulation of 4 groups per step (was 1) for less noisy updates.
- Separate LR for the freshly initialised scoring head (1e-4) vs the pretrained encoder blocks (1.5e-5).
- EMA (0.995) of trainable student weights, used for Dev evaluation and the saved checkpoint.
- 6 epochs / patience 3 (was 4 / 2); finer Dev fusion-weight grid (`ALPHA_GRID`).


## 1. Runtime, checkpoint discovery and Phase-7 data

In [1]:
import os,sys,glob,json,time,math,random,re,gc,zipfile,pickle,hashlib,warnings,heapq
from contextlib import contextmanager
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from pathlib import Path
from collections import Counter
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

SEED=731
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type=="cuda", "Select a Kaggle GPU (T4 recommended)."
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
torch.set_float32_matmul_precision("high")
def stamp(): return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%d %H:%M:%S IST")
OUTPUT_DIR=Path("/kaggle/working/phase7_model1_to_model3_kd")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

# Explicitly identify the two checkpoints; never accidentally load the extra .pt file.
all_pt=glob.glob("/kaggle/input/**/*.pt",recursive=True)
m3_candidates=[p for p in all_pt if "encdec_full_best" in os.path.basename(p).lower()]
if not m3_candidates:
    m3_candidates=[p for p in all_pt if "encdec_full" in os.path.basename(p).lower() and "best" in os.path.basename(p).lower()]
if not m3_candidates: raise FileNotFoundError("Model 3 encdec_full_best*.pt not found. Attach phase2_model3_ckpts.")
M3_CKPT=sorted(m3_candidates,key=lambda p:("phase2_model3_ckpts" not in p.lower(),len(p)))[0]

teacher_candidates=[p for p in all_pt if os.path.basename(p).lower()=="best_dev_f1.pt" and "model1" in p.lower()]
if not teacher_candidates:
    teacher_candidates=[p for p in all_pt if os.path.basename(p).lower()=="best_dev_f1.pt" and "phase7" in p.lower()]
if not teacher_candidates:
    teacher_candidates=[p for p in all_pt if os.path.basename(p).lower()=="best_dev_f1.pt"]
if not teacher_candidates:
    raise FileNotFoundError("Model 1 teacher best_dev_f1.pt not found. Attach the Experiment 5 output/checkpoint dataset.")
M1_CKPT=sorted(teacher_candidates,key=lambda p:("phase7_model1_full_ft" not in p.lower(),len(p)))[0]

# Model 3 tokenizer shown in phase2_model3_ckpts. Model 1 tokenizer must be selected separately.
tok_files=glob.glob("/kaggle/input/**/*.json",recursive=True)
def is_tokenizer(p): return os.path.basename(p).lower().startswith("tokenizer") and p.lower().endswith(".json")
tok_files=[p for p in tok_files if is_tokenizer(p)]
m3_toks=[p for p in tok_files if os.path.dirname(p)==os.path.dirname(M3_CKPT)]
if not m3_toks: m3_toks=[p for p in tok_files if "phase2_model3_ckpts" in p.lower()]
if not m3_toks: raise FileNotFoundError("Model 3 tokenizer not found beside phase2_model3_ckpts.")
M3_TOKENIZER=sorted(m3_toks)[0]
m1_toks=[p for p in tok_files if os.path.dirname(p)==os.path.dirname(M1_CKPT)]
if not m1_toks: m1_toks=[p for p in tok_files if "model1" in p.lower() or "phase1model-wts" in p.lower()]
if not m1_toks:
    # In these Phase-1 runs tokenizers are byte-level BPE; only fall back if unambiguous.
    if len(tok_files)==1: m1_toks=tok_files
    else: raise FileNotFoundError("Cannot identify Model 1 tokenizer. Attach its tokenizer JSON beside teacher checkpoint or set M1_TOKENIZER manually.")
M1_TOKENIZER=sorted(m1_toks)[0]
print("Teacher:",M1_CKPT,"\nStudent:",M3_CKPT,"\nTeacher tokenizer:",M1_TOKENIZER,"\nStudent tokenizer:",M3_TOKENIZER)

REPO_ID="Exploration-Lab/CS-781-Capstone"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception: HF_TOKEN=None
def load_hf_json(rel):
    p=hf_hub_download(repo_id=REPO_ID,filename=rel,repo_type="dataset",token=HF_TOKEN)
    with open(p,encoding="utf-8") as f: return json.load(f)
train_queries=load_hf_json("Phase_7_PCR/Train/train.json")
train_candidates=load_hf_json("Phase_7_PCR/Train/train_candidates.json")
dev_queries=load_hf_json("Phase_7_PCR/Dev/dev.json")
dev_candidates=load_hf_json("Phase_7_PCR/Dev/dev_candidates.json")
test_queries=load_hf_json("Phase_7_PCR/Test/test.json")
test_candidates=load_hf_json("Phase_7_PCR/Test/test_candidates.json")
def normalize(d): return {str(k):v for k,v in d.items()}
train_queries=normalize(train_queries);train_candidates=normalize(train_candidates)
dev_queries=normalize(dev_queries);dev_candidates=normalize(dev_candidates)
test_queries=normalize(test_queries);test_candidates=normalize(test_candidates)
def record_text(r):
    v=r.get("text",[])
    return " ".join(map(str,v)) if isinstance(v,list) else str(v)
print("Loaded splits:",len(train_queries),len(dev_queries),len(test_queries))

Teacher: /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/best_dev_f1.pt 
Student: /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt 
Teacher tokenizer: /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/tokenizer (1).json 
Student tokenizer: /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/tokenizer (1).json


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (2 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

Loaded splits: 827 118 237


## 2. Model configuration and independent LegalBPE tokenizers

In [2]:
# Model-specific architecture/configuration. Tokenizers are loaded independently.
M1_D_MODEL=768; M1_N_LAYERS=14; M1_N_HEADS=12; M1_FFN_DIM=3072; M1_DROPOUT=0.1; M1_MAX_POSITIONS=2048
D_MODEL=512; N_ENCODER_LAYERS=9; N_DECODER_LAYERS=9; N_HEADS=8; FFN_DIM=2048; DROPOUT=0.0
MAX_ENCODER_POSITIONS=2048; MAX_DECODER_POSITIONS=1536
PAIR_MAX_LEN=320; QUERY_TOKENS=112; CANDIDATE_TOKENS=204
RERANK_TOP_N=100; TRAIN_NEGATIVES=7; HARD_NEGATIVE_POOL=100; TOP_K=7   # more / harder negatives per group
EPOCHS=6; LR=1.5e-5; HEAD_LR=1.0e-4; WEIGHT_DECAY=0.01; WARMUP_FRAC=0.06   # head is freshly initialised -> higher LR
KD_TEMPERATURE=2.0; LAMBDA_KD=0.50; LAMBDA_HARD=0.35; LAMBDA_PAIR=0.15
BATCH_GROUPS=4; EVAL_BATCH_SIZE=12; GRAD_CLIP=1.0   # 4 query-groups per optimizer step (gradient accumulation)
TEACHER_GROUPS=3      # distinct (positive, negatives) groups cached per query; epoch e uses group e%TEACHER_GROUPS
EMA_DECAY=0.995       # EMA of trainable student weights, used for evaluation / checkpoint
PATIENCE=3
ALPHA_GRID=sorted(set([0.,0.01]+[round(x,2) for x in np.arange(0.02,1.001,0.02)]+[1.25,1.5,2.0,2.5,3.0,4.0]))   # finer Dev fusion search


# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok


# Load each tokenizer independently; same vocabulary size does not imply identical token IDs.
m1_tokenizer=LegalBPE.load_json(M1_TOKENIZER)
m3_tokenizer=LegalBPE.load_json(M3_TOKENIZER)
tokenizer=m3_tokenizer  # build_model() expects the student tokenizer global
print("Tokenizer sizes:",len(m1_tokenizer),len(m3_tokenizer))
assert len(m1_tokenizer)==24576 and len(m3_tokenizer)==24576

Tokenizer sizes: 24576 24576


## 3. Model 1 teacher architecture and frozen checkpoint

In [3]:
GRAD_CHECKPOINTING=False
D_MODEL= M1_D_MODEL
N_LAYERS=M1_N_LAYERS
N_HEADS=M1_N_HEADS
FFN_DIM=M1_FFN_DIM
DROPOUT=M1_DROPOUT
MAX_POSITIONS=M1_MAX_POSITIONS
# Cell 9 — Optimized ~120M Prefix Encoder LM with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class PrefixSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B,T,C = x.shape
        qkv = self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v = qkv.unbind(dim=2)
        q = q.transpose(1,2)
        k = k.transpose(1,2)
        v = v.transpose(1,2)

        if past_kv is not None:
            pk,pv = past_kv
            k = torch.cat([pk,k], dim=2)
            v = torch.cat([pv,v], dim=2)

        y = F.scaled_dot_product_attention(
            q,k,v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1,2).contiguous().view(B,T,C)
        y = self.out_proj(y)

        new_kv = (k,v) if use_cache else None
        return y, new_kv


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixSelfAttention(d_model,n_heads,dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model,ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim,d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        a,new_kv = self.attn(
            self.ln1(x),
            attn_mask=attn_mask,
            past_kv=past_kv,
            use_cache=use_cache,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x,new_kv


class PrefixEncoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size,d_model)
        self.pos = SinusoidalPositionalEncoding(d_model,max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            EncoderBlock(d_model,n_heads,ffn_dim,dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        # Tied output projection: no second vocab-sized matrix.
        self.apply(self._init_weights)

        # GPT-style residual projection scaling improves deep-model stability.
        scaled_std = 0.02 / math.sqrt(2*n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    @staticmethod
    def prefix_mask(T, prefix_len, device):
        # True means attention is allowed for SDPA boolean mask.
        q = torch.arange(T, device=device)[:,None]
        k = torch.arange(T, device=device)[None,:]
        p = int(prefix_len)
        allowed = ((q < p) & (k < p)) | ((q >= p) & (k <= q))
        return allowed[None,None,:,:]

    def forward_hidden(self, input_ids, prefix_len=None, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        # position_ids / key_valid are only used for LEFT-PADDED BATCHED generation; when both are None the
        # behaviour is exactly the original (training path unchanged).
        B,T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len+T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len+T, device=input_ids.device
            )[None,:].expand(B,T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        # Full training sequence uses the prefix-LM mask.
        # Initial generation prefill has prefix_len == T, therefore it is fully bidirectional.
        # Cached one-token decoding needs no mask because only past/current KVs exist.
        if key_valid is not None:
            # (B,1,1,S) bool, True = real token. Prompt is bidirectional; padding is never attended to.
            attn_mask = key_valid[:,None,None,:]
        elif past_key_values is None and prefix_len is not None and int(prefix_len) < T:
            attn_mask = self.prefix_mask(T,prefix_len,input_ids.device)
        else:
            attn_mask = None

        new_cache = [] if use_cache else None

        for i,block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            if (GRAD_CHECKPOINTING and self.training and torch.is_grad_enabled()
                    and not use_cache and past_key_values is None):
                # Save activation memory while retaining gradients through every block.
                x = grad_checkpoint(
                    lambda z, _block=block, _mask=attn_mask: _block(
                        z, attn_mask=_mask, past_kv=None, use_cache=False
                    )[0],
                    x,
                    use_reentrant=False
                )
                kv = None
            else:
                x,kv = block(
                    x,
                    attn_mask=attn_mask,
                    past_kv=past,
                    use_cache=use_cache,
                )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x,new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)





class TeacherPairScorer(nn.Module):
    def __init__(self,backbone):
        super().__init__()
        self.backbone=backbone
        self.dropout=nn.Dropout(0.12)
        self.score=nn.Sequential(nn.Linear(M1_D_MODEL,384),nn.GELU(),nn.Dropout(0.12),nn.Linear(384,1))
    def forward(self,input_ids,valid):
        h,_=self.backbone.forward_hidden(input_ids,prefix_len=input_ids.shape[1],key_valid=valid)
        mask=valid.unsqueeze(-1).to(h.dtype)
        pooled=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
        return self.score(self.dropout(pooled.float())).squeeze(-1)

teacher=PrefixEncoderLM(vocab_size=len(m1_tokenizer),d_model=M1_D_MODEL,n_layers=M1_N_LAYERS,
    n_heads=M1_N_HEADS,ffn_dim=M1_FFN_DIM,dropout=M1_DROPOUT,max_positions=M1_MAX_POSITIONS).to(DEVICE)
teacher_head=TeacherPairScorer(teacher).to(DEVICE)
teacher_obj=torch.load(M1_CKPT,map_location="cpu",weights_only=False)
if "model" not in teacher_obj or "scorer" not in teacher_obj:
    raise KeyError(f"Teacher checkpoint must contain model and scorer. Keys={list(teacher_obj)[:20]}")
teacher.load_state_dict(teacher_obj["model"],strict=True)
teacher_head.score.load_state_dict(teacher_obj["scorer"],strict=True)
del teacher_obj
teacher.eval(); teacher_head.eval()
for p in teacher.parameters(): p.requires_grad=False
for p in teacher_head.parameters(): p.requires_grad=False
print("Teacher loaded and frozen | params:",sum(p.numel() for p in teacher.parameters())/1e6,"M")

Teacher loaded and frozen | params: 118.106112 M


## 4. Model 3 student architecture and Phase-1 checkpoint

In [4]:
D_MODEL=512
N_HEADS=8
FFN_DIM=2048
DROPOUT=0.0
# Cell 9 — ~79M Encoder–Decoder Transformer (SDPA, cross-attention, decode KV cache)
# v2 changes vs v1:
#   * token embeddings are scaled by sqrt(d_model) before the sinusoidal positions are added
#     (v1 added unit-scale positions to 0.02-std embeddings => positions drowned the tokens)
#   * encoder positions are derived from the padding mask (so padded / doc-masked rows get positions 0..n-1)
#   * decoder self-attention handles "cache + several new tokens" correctly (needed for prompt priming)
#   * residual-branch init uses the true branch count (3 per decoder layer)
#   * build_model() so Cell 11 always starts from fresh weights (no more "rerun Cell 9" step)
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class EncoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, key_valid=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        attn_mask = None
        if key_valid is not None:
            attn_mask = key_valid[:, None, None, :]  # True = valid encoder key

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class DecoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        mask, is_causal = None, True
        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
            if T == 1:
                is_causal = False                      # the new token may see every cached key
            else:
                S = k.size(2)
                P = S - T                              # query i sits at global position P+i
                mask = torch.ones(T, S, device=x.device).tril(diagonal=P).bool()
                is_causal = False

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)


class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.kv_proj = nn.Linear(d_model, 2 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, memory, memory_valid=None, cached_kv=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if cached_kv is None:
            S = memory.size(1)
            kv = self.kv_proj(memory).view(B, S, 2, self.n_heads, self.head_dim)
            k, v = kv.unbind(dim=2)
            k, v = k.transpose(1, 2), v.transpose(1, 2)
        else:
            k, v = cached_kv

        attn_mask = None
        if memory_valid is not None:
            attn_mask = memory_valid[:, None, None, :]

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = EncoderSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_valid=None):
        x = x + self.drop(self.attn(self.ln1(x), key_valid=key_valid))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = DecoderSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_heads, dropout)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)

    def forward(
        self, x, memory, memory_valid=None,
        past_self_kv=None, cached_cross_kv=None, use_cache=False
    ):
        a, new_self = self.self_attn(
            self.ln1(x), past_kv=past_self_kv, use_cache=use_cache
        )
        x = x + self.drop(a)

        c, new_cross = self.cross_attn(
            self.ln2(x),
            memory,
            memory_valid=memory_valid,
            cached_kv=cached_cross_kv,
            use_cache=use_cache,
        )
        x = x + self.drop(c)
        x = x + self.ff(self.ln3(x))
        return x, new_self, new_cross


class EncoderDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_encoder_layers, n_decoder_layers,
        n_heads, ffn_dim, dropout, max_encoder_positions, max_decoder_positions
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.emb_scale = math.sqrt(d_model)
        self.max_encoder_positions = max_encoder_positions
        self.max_decoder_positions = max_decoder_positions

        # Shared embedding across encoder input, decoder input and output projection.
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.enc_pos = SinusoidalPositionalEncoding(d_model, max_encoder_positions)
        self.dec_pos = SinusoidalPositionalEncoding(d_model, max_decoder_positions)
        self.drop = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([
            EncoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_encoder_layers)
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_decoder_layers)
        ])
        self.enc_ln = nn.LayerNorm(d_model)
        self.dec_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-2 style depth scaling: std = 0.02 / sqrt(#residual branches)
        enc_scaled = 0.02 / math.sqrt(2 * n_encoder_layers)   # attn + ffn
        dec_scaled = 0.02 / math.sqrt(3 * n_decoder_layers)   # self-attn + cross-attn + ffn
        for block in self.encoder:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=enc_scaled)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=enc_scaled)
        for block in self.decoder:
            nn.init.normal_(block.self_attn.out_proj.weight, mean=0.0, std=dec_scaled)
            nn.init.normal_(block.cross_attn.out_proj.weight, mean=0.0, std=dec_scaled)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=dec_scaled)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def encode(self, input_ids, key_valid=None, position_ids=None):
        B, S = input_ids.shape
        if S > self.max_encoder_positions:
            raise ValueError(f"encoder length {S} > {self.max_encoder_positions}")

        if position_ids is None:
            if key_valid is None:
                position_ids = torch.arange(S, device=input_ids.device)[None, :].expand(B, S)
            else:
                # valid tokens always get positions 0..n-1, wherever the padding / masked tokens sit
                position_ids = (key_valid.long().cumsum(1) - 1).clamp(min=0)

        x = self.token_emb(input_ids) * self.emb_scale \
            + self.enc_pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)
        for block in self.encoder:
            x = block(x, key_valid=key_valid)
        return self.enc_ln(x)

    def decode_hidden(
        self, decoder_ids, memory, memory_valid=None,
        past_self_key_values=None, cross_key_values=None,
        use_cache=False, position_ids=None
    ):
        B, T = decoder_ids.shape
        past_len = 0
        if past_self_key_values is not None:
            past_len = past_self_key_values[0][0].size(2)

        if past_len + T > self.max_decoder_positions:
            raise ValueError(
                f"decoder length {past_len + T} > {self.max_decoder_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=decoder_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(decoder_ids) * self.emb_scale \
            + self.dec_pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_self, new_cross = ([] if use_cache else None), ([] if use_cache else None)

        for i, block in enumerate(self.decoder):
            ps = None if past_self_key_values is None else past_self_key_values[i]
            ck = None if cross_key_values is None else cross_key_values[i]
            x, s_kv, c_kv = block(
                x, memory, memory_valid=memory_valid,
                past_self_kv=ps, cached_cross_kv=ck, use_cache=use_cache
            )
            if use_cache:
                new_self.append(s_kv)
                new_cross.append(c_kv)

        return self.dec_ln(x), new_self, new_cross

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_model():
    return EncoderDecoderLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_encoder_layers=N_ENCODER_LAYERS,
        n_decoder_layers=N_DECODER_LAYERS,
        n_heads=N_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_encoder_positions=MAX_ENCODER_POSITIONS,
        max_decoder_positions=MAX_DECODER_POSITIONS,
    ).to(DEVICE)


class StudentPairScorer(nn.Module):
    """Model 3 cross-encoder scoring head: encoder over [BOS] query [EOS] candidate [EOS],
    masked mean-pool, MLP -> one relevance logit (same formulation as the teacher scorer)."""
    def __init__(self,backbone):
        super().__init__()
        self.backbone=backbone
        self.dropout=nn.Dropout(0.10)
        self.score=nn.Sequential(nn.Linear(D_MODEL,256),nn.GELU(),nn.Dropout(0.10),nn.Linear(256,1))
    def forward(self,input_ids,valid):
        h=self.backbone.encode(input_ids,key_valid=valid)
        mask=valid.unsqueeze(-1).to(h.dtype)
        pooled=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
        return self.score(self.dropout(pooled.float())).squeeze(-1)

student=build_model().to(DEVICE)
student_obj=torch.load(M3_CKPT,map_location="cpu",weights_only=False)
student_state=student_obj.get("model",student_obj.get("state_dict",student_obj))
missing,unexpected=student.load_state_dict(student_state,strict=False)
if missing or unexpected:
    raise RuntimeError(f"Model 3 checkpoint mismatch. missing={missing[:12]}, unexpected={unexpected[:12]}")
del student_obj,student_state
student_head=StudentPairScorer(student).to(DEVICE)
# Match the successful Model 3 cross-encoder experiment: train last 3 encoder blocks + scoring head.
for p in student.parameters(): p.requires_grad=False
for block in student.encoder[-3:]:
    for p in block.parameters(): p.requires_grad=True
for p in student.enc_ln.parameters(): p.requires_grad=True
# IMPORTANT: student_head contains the student backbone as a child module.
# Do NOT unfreeze student_head.parameters() here; that would silently unfreeze all 9 encoder
# and all 9 decoder blocks. Only the newly initialized scoring MLP should be fully trainable.
for p in student_head.score.parameters(): p.requires_grad=True
print("Student loaded | params:",sum(p.numel() for p in student.parameters())/1e6,"M")
trainable_backbone=sum(p.numel() for p in student.parameters() if p.requires_grad)
trainable_head=sum(p.numel() for p in student_head.score.parameters())
print("Trainable backbone params:",trainable_backbone/1e6,"M | scoring head:",trainable_head/1e6,"M")
print("Trainable student params:",(trainable_backbone+trainable_head)/1e6,"M")
assert all(not p.requires_grad for p in student.decoder.parameters()), "Decoder must remain frozen/unused for the Model 3 cross-encoder."
assert all(p.requires_grad for block in student.encoder[-3:] for p in block.parameters()), "Last three encoder blocks should be trainable."
assert all(not p.requires_grad for block in student.encoder[:-3] for p in block.parameters()), "First six encoder blocks should remain frozen."

Student loaded | params: 78.792704 M
Trainable backbone params: 9.458176 M | scoring head: 0.131585 M
Trainable student params: 9.589761 M


## 5. BM25 candidate mining, pair construction and teacher target cache

In [5]:
def bm25_clean(text):
    # Keep legal section numbers and alphanumeric case references.
    return re.sub(r"\s+", " ", str(text)).strip()

class BM25Index:
    def __init__(self, docs, k1=1.35, b=0.72, max_features=300000):
        self.ids = list(docs.keys())
        self.texts = [bm25_clean(record_text(docs[x])) for x in self.ids]
        self.vectorizer = CountVectorizer(
            lowercase=True, stop_words="english",
            token_pattern=r"(?u)\b[a-zA-Z0-9][a-zA-Z0-9./_-]{1,}\b",
            ngram_range=(1, 2), max_features=max_features,
            min_df=1, binary=False, dtype=np.float32
        )
        counts = self.vectorizer.fit_transform(self.texts).tocsr()
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
        self.avgdl = float(self.doc_len.mean()) if len(self.doc_len) else 1.0
        df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(np.float32)
        n = len(self.ids)
        self.idf = np.log(1.0 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        # Build BM25 weights directly on nonzero entries; never densify the
        # document-term matrix (which can exceed several GB for legal corpora).
        row_ids = np.repeat(np.arange(counts.shape[0]), np.diff(counts.indptr))
        tf = counts.data.astype(np.float32, copy=False)
        norm = k1 * (1.0 - b + b * self.doc_len[row_ids] / max(self.avgdl, 1.0))
        weighted = self.idf[counts.indices] * (tf * (k1 + 1.0) / (tf + norm))
        self.matrix = sparse.csr_matrix(
            (weighted.astype(np.float32), counts.indices.copy(), counts.indptr.copy()),
            shape=counts.shape
        )
        self.k1, self.b = k1, b
        self.id_to_row = {x:i for i,x in enumerate(self.ids)}

    def search(self, query_record, top_n=100, exclude_id=None):
        q = self.vectorizer.transform([bm25_clean(record_text(query_record))])
        scores = np.asarray((self.matrix @ q.T).toarray()).ravel()
        if exclude_id in self.id_to_row:
            scores[self.id_to_row[exclude_id]] = -1e9
        k = min(top_n, len(scores))
        idx = np.argpartition(-scores, k-1)[:k]
        idx = idx[np.argsort(-scores[idx])]
        return [(self.ids[i], float(scores[i])) for i in idx]


print("Building Train BM25 index:",stamp())
bm25_train=BM25Index(train_candidates)
train_examples=[]
for qid,qrec in train_queries.items():
    pos=[str(x) for x in qrec.get("relevant_candidates",[]) if str(x) in train_candidates and str(x)!=qid]
    if not pos: continue
    ranked=bm25_train.search(qrec,top_n=HARD_NEGATIVE_POOL,exclude_id=qid)
    neg=[cid for cid,_ in ranked if cid not in set(pos)]
    if len(neg)<TRAIN_NEGATIVES:
        neg += [cid for cid in train_candidates if cid not in set(pos) and cid!=qid and cid not in neg]
    if neg: train_examples.append({"qid":qid,"positive_ids":pos,"negative_ids":neg[:max(40,4*TRAIN_NEGATIVES)]})
assert train_examples
print("Train groups:",len(train_examples))

def encode_side(tok,record,limit):
    ids=tok.encode(record_text(record))
    if not ids: ids=[tok.unk_token_id]
    if len(ids)>limit:
        h=int(limit*0.68); ids=ids[:h]+ids[-(limit-h):]
    return ids

def pair_ids(tok,qrec,crec,q_lim,c_lim,max_len):
    q=encode_side(tok,qrec,q_lim); c=encode_side(tok,crec,c_lim)
    return ([tok.bos_token_id]+q+[tok.eos_token_id]+c+[tok.eos_token_id])[:max_len]

def collate_ids(rows,tok):
    width=max(map(len,rows))
    x=torch.full((len(rows),width),tok.pad_token_id,dtype=torch.long)
    valid=torch.zeros((len(rows),width),dtype=torch.bool)
    for i,row in enumerate(rows):
        x[i,:len(row)]=torch.tensor(row,dtype=torch.long); valid[i,:len(row)]=True
    return x.to(DEVICE),valid.to(DEVICE)

@torch.no_grad()
def teacher_group_logits(qrec,cids):
    teacher.eval(); teacher_head.eval()
    rows=[pair_ids(m1_tokenizer,qrec,train_candidates[c],128,300,448) for c in cids]
    x,v=collate_ids(rows,m1_tokenizer)
    with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=True):
        z=teacher_head(x,v)
    return z.float().cpu().numpy()

# Freeze teacher targets on TEACHER_GROUPS different (positive + BM25 hard-negative) groups per query.
# Each group mixes the hardest BM25 negatives with random pool negatives. Cache is flat and reusable.
teacher_cache_path=OUTPUT_DIR/"teacher_train_logits_v2.pkl"
# Cache signature prevents stale teacher logits from being reused after changing checkpoints,
# candidate mining, sampling settings, or the underlying Train split.
_cache_payload={
    "teacher_ckpt":os.path.abspath(M1_CKPT),
    "teacher_size":os.path.getsize(M1_CKPT),
    "teacher_mtime":os.path.getmtime(M1_CKPT),
    "m1_tokenizer":os.path.abspath(M1_TOKENIZER),
    "m1_tok_size":os.path.getsize(M1_TOKENIZER),
    "train_query_ids":sorted(train_queries),
    "train_candidate_count":len(train_candidates),
    "train_examples_sha256":hashlib.sha256(json.dumps(train_examples,sort_keys=True).encode()).hexdigest(),
    "groups":TEACHER_GROUPS,"negatives":TRAIN_NEGATIVES,"pool":HARD_NEGATIVE_POOL,
    "seed":SEED
}
CACHE_SIGNATURE=hashlib.sha256(json.dumps(_cache_payload,sort_keys=True).encode()).hexdigest()
teacher_cache={}
if teacher_cache_path.exists():
    try:
        cached_obj=pickle.load(open(teacher_cache_path,"rb"))
        if isinstance(cached_obj,dict) and cached_obj.get("signature")==CACHE_SIGNATURE:
            teacher_cache=cached_obj.get("cache",{})
            print("Loaded matching cached teacher groups:",len(teacher_cache))
        else:
            print("Teacher cache signature changed; rebuilding targets.")
    except Exception as e:
        print("Could not reuse teacher cache; rebuilding:",repr(e))
def _cache_ok(c):
    return len(c)==len(train_examples) and all(ex["qid"] in c and len(c[ex["qid"]])==TEACHER_GROUPS for ex in train_examples)
if not _cache_ok(teacher_cache):
    teacher_cache={}
    for gi,ex in enumerate(train_examples,1):
        q=train_queries[ex["qid"]]
        positives=list(ex["positive_ids"]); random.shuffle(positives)
        pool=ex["negative_ids"]; hard=pool[:15]
        groups=[]
        for g in range(TEACHER_GROUPS):
            pos=positives[g%len(positives)]
            picks=random.sample(hard,k=min(3,len(hard),TRAIN_NEGATIVES))
            rest=[c for c in pool if c not in picks]
            picks+=random.sample(rest,k=min(TRAIN_NEGATIVES-len(picks),len(rest)))
            cids=[pos]+picks
            groups.append((cids,teacher_group_logits(q,cids).astype(np.float32)))
        teacher_cache[ex["qid"]]=groups
        if gi%100==0 or gi==len(train_examples): print(f"Teacher targets {gi}/{len(train_examples)}")
    tmp=teacher_cache_path.with_suffix(".tmp")
    with open(tmp,"wb") as f:
        pickle.dump({"signature":CACHE_SIGNATURE,"cache":teacher_cache},f,pickle.HIGHEST_PROTOCOL)
    os.replace(tmp,teacher_cache_path)
print("Teacher target cache ready:",len(teacher_cache),"queries x",TEACHER_GROUPS,"groups")


Building Train BM25 index: 2026-11-07 09:50:03 IST
Train groups: 817
Teacher targets 100/817
Teacher targets 200/817
Teacher targets 300/817
Teacher targets 400/817
Teacher targets 500/817
Teacher targets 600/817
Teacher targets 700/817
Teacher targets 800/817
Teacher targets 817/817
Teacher target cache ready: 817 queries x 3 groups


## 6. KD training

In [6]:
def student_group_logits(qrec,cids):
    rows=[pair_ids(m3_tokenizer,qrec,train_candidates[c],QUERY_TOKENS,CANDIDATE_TOKENS,PAIR_MAX_LEN) for c in cids]
    x,v=collate_ids(rows,m3_tokenizer)
    return student_head(x,v)

def kd_group_loss(student_logits,teacher_logits):
    T=KD_TEMPERATURE
    target=F.softmax(torch.as_tensor(teacher_logits,device=DEVICE,dtype=torch.float32)/T,dim=0)
    logp=F.log_softmax(student_logits.float()/T,dim=0)
    kd=F.kl_div(logp,target,reduction="sum")*(T*T)
    hard=F.cross_entropy(student_logits.float().unsqueeze(0),torch.zeros(1,dtype=torch.long,device=DEVICE))
    # Positive is at index 0; enforce it above each negative.
    pair=F.softplus(-(student_logits[0]-student_logits[1:])).mean() if len(student_logits)>1 else student_logits.new_tensor(0.)
    return LAMBDA_KD*kd+LAMBDA_HARD*hard+LAMBDA_PAIR*pair,{"kd":kd.detach(),"hard":hard.detach(),"pair":pair.detach()}

# Optimizer over trainable student parameters only.
# (student_head.parameters() would also list the whole backbone again, so the head is addressed via .score)
enc_params=[p for p in student.parameters() if p.requires_grad]
head_params=list(student_head.score.parameters())
train_params=enc_params+head_params
optimizer=torch.optim.AdamW([{"params":enc_params,"lr":LR},{"params":head_params,"lr":HEAD_LR}],weight_decay=WEIGHT_DECAY)
steps_per_epoch=math.ceil(len(train_examples)/BATCH_GROUPS)
total_steps=steps_per_epoch*EPOCHS

# EMA of trainable weights (smoother, usually better Dev/Test than the raw last step)
ema_state=[(p,p.detach().clone()) for p in train_params]; ema_steps=0
def ema_update():
    global ema_steps
    ema_steps+=1; d=min(EMA_DECAY,(1+ema_steps)/(10+ema_steps))
    with torch.no_grad():
        for p,e in ema_state: e.mul_(d).add_(p.detach(),alpha=1-d)
@contextmanager
def use_ema():
    backup=[p.detach().clone() for p,_ in ema_state]
    with torch.no_grad():
        for p,e in ema_state: p.copy_(e)
    try: yield
    finally:
        with torch.no_grad():
            for (p,_),b in zip(ema_state,backup): p.copy_(b)
warmup=max(1,int(total_steps*WARMUP_FRAC))
def lr_scale(step):
    if step<warmup: return (step+1)/warmup
    return max(0.1,0.5*(1+math.cos(math.pi*(step-warmup)/max(1,total_steps-warmup))))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_scale)
scaler=torch.cuda.amp.GradScaler(enabled=True)
history=[]; best_dev=-1.; best_epoch=0; patience=0

# BM25 Dev index and evaluation helpers.
bm25_dev=BM25Index(dev_candidates)
def metrics(queries,preds):
    tp=fp=fn=0
    for qid,q in queries.items():
        g=set(map(str,q.get("relevant_candidates",[]))); p=set(preds.get(qid,[])[:7])
        tp+=len(g&p);fp+=len(p-g);fn+=len(g-p)
    pr=tp/max(1,tp+fp);rc=tp/max(1,tp+fn);f=2*pr*rc/max(1e-12,pr+rc)
    return {"micro_f1":f,"micro_precision":pr,"micro_recall":rc,"tp":tp,"fp":fp,"fn":fn}

@torch.no_grad()
def evaluate_student(queries,candidates,index,split):
    student.eval();student_head.eval(); cache={}
    for qi,(qid,qrec) in enumerate(queries.items(),1):
        ranked=index.search(qrec,top_n=RERANK_TOP_N,exclude_id=qid)
        ids=[c for c,_ in ranked]
        rows=[pair_ids(m3_tokenizer,qrec,candidates[c],QUERY_TOKENS,CANDIDATE_TOKENS,PAIR_MAX_LEN) for c in ids]
        vals=[]
        for st in range(0,len(rows),EVAL_BATCH_SIZE):
            x,v=collate_ids(rows[st:st+EVAL_BATCH_SIZE],m3_tokenizer)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=True):
                z=student_head(x,v)
            vals.extend(z.float().cpu().tolist())
        lex=np.asarray([s for _,s in ranked],np.float32)
        neu=np.asarray(vals,np.float32)
        lex=(lex-lex.mean())/(lex.std()+1e-6); neu=(neu-neu.mean())/(neu.std()+1e-6)
        cache[qid]=(ids,lex,neu)
        if qi%25==0 or qi==len(queries): print(f"{split}: {qi}/{len(queries)}")
    return cache

def predictions_from_cache(cache,alpha):
    out={}
    for q,(ids,lex,neu) in cache.items():
        order=np.argsort(-(neu+alpha*lex),kind="stable")
        out[q]=[ids[i] for i in order[:7]]
    return out

for epoch in range(EPOCHS):
    student.train();student_head.train()
    random.shuffle(train_examples)
    losses=[]; kdvals=[]; hardvals=[]; pairvals=[]
    optimizer.zero_grad(set_to_none=True); accum=0
    for gi,ex in enumerate(train_examples,1):
        groups=teacher_cache.get(ex["qid"])
        if groups is None: continue
        cids,tlogits=groups[epoch%len(groups)]
        q=train_queries[ex["qid"]]
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=True):
            slogits=student_group_logits(q,cids)
            loss,parts=kd_group_loss(slogits,tlogits)
        scaler.scale(loss/BATCH_GROUPS).backward(); accum+=1
        if accum==BATCH_GROUPS or gi==len(train_examples):
            scaler.unscale_(optimizer)
            # Correctly average the final partial accumulation window as well.
            if accum < BATCH_GROUPS:
                correction=BATCH_GROUPS/accum
                for p in train_params:
                    if p.grad is not None: p.grad.mul_(correction)
            torch.nn.utils.clip_grad_norm_(train_params,GRAD_CLIP)
            scaler.step(optimizer);scaler.update();scheduler.step()
            optimizer.zero_grad(set_to_none=True); accum=0
            ema_update()
        losses.append(float(loss.detach().cpu()))
        kdvals.append(float(parts["kd"].cpu()));hardvals.append(float(parts["hard"].cpu()));pairvals.append(float(parts["pair"].cpu()))
        if gi%100==0 or gi==len(train_examples): print(f"Epoch {epoch+1} groups {gi}/{len(train_examples)} loss={np.mean(losses):.4f}")
    with use_ema():     # evaluate and checkpoint the EMA weights
        dev_cache=evaluate_student(dev_queries,dev_candidates,bm25_dev,"Dev")
        best_a=0.;best_m=metrics(dev_queries,predictions_from_cache(dev_cache,0.))
        for a in ALPHA_GRID:
            m=metrics(dev_queries,predictions_from_cache(dev_cache,a))
            if m["micro_f1"]>best_m["micro_f1"]:best_a,best_m=a,m
        bm25_pred={q:[c for c,_ in bm25_dev.search(rec,top_n=7,exclude_id=q)] for q,rec in dev_queries.items()}
        bm25_m=metrics(dev_queries,bm25_pred)
        if bm25_m["micro_f1"]>best_m["micro_f1"]: best_m=bm25_m; best_a=None
        row={"epoch":epoch+1,"loss":float(np.mean(losses)),"kd_loss":float(np.mean(kdvals)),
             "hard_loss":float(np.mean(hardvals)),"pair_loss":float(np.mean(pairvals)),
             "dev":best_m,"alpha":best_a}
        history.append(row);print("EPOCH RESULT:",json.dumps(row,indent=2))
        if best_m["micro_f1"]>best_dev:
            best_dev=best_m["micro_f1"];best_epoch=epoch+1;patience=0
            torch.save({"student":student.state_dict(),"head":student_head.score.state_dict(),
                        "epoch":best_epoch,"dev":best_m,"alpha":best_a,"config":{
                        "temperature":KD_TEMPERATURE,"lambda_kd":LAMBDA_KD,"lambda_hard":LAMBDA_HARD,
                        "lambda_pair":LAMBDA_PAIR,"ema_decay":EMA_DECAY}},OUTPUT_DIR/"best_student_kd.pt")
        else:
            patience+=1
    with open(OUTPUT_DIR/"kd_training_history.json","w") as f:json.dump(history,f,indent=2)
    if patience>=PATIENCE: print("Early stopping.");break
print("Best epoch:",best_epoch,"best Dev F1:",best_dev)


/tmp/ipykernel_50/3944762940.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=True)


Epoch 1 groups 100/817 loss=0.8325
Epoch 1 groups 200/817 loss=0.8325
Epoch 1 groups 300/817 loss=0.8325
Epoch 1 groups 400/817 loss=0.8325
Epoch 1 groups 500/817 loss=0.8325
Epoch 1 groups 600/817 loss=0.8325
Epoch 1 groups 700/817 loss=0.8325
Epoch 1 groups 800/817 loss=0.8325
Epoch 1 groups 817/817 loss=0.8325
Dev: 25/118
Dev: 50/118
Dev: 75/118
Dev: 100/118
Dev: 118/118
EPOCH RESULT: {
  "epoch": 1,
  "loss": 0.8324715615486136,
  "kd_loss": 0.0015615288309543194,
  "hard_loss": 2.0792446387441537,
  "pair_loss": 0.6930345392431448,
  "dev": {
    "micro_f1": 0.330183435241801,
    "micro_precision": 0.3595641646489104,
    "micro_recall": 0.3052415210688592,
    "tp": 297,
    "fp": 529,
    "fn": 676
  },
  "alpha": null
}
Epoch 2 groups 100/817 loss=0.8320
Epoch 2 groups 200/817 loss=0.8322
Epoch 2 groups 300/817 loss=0.8321
Epoch 2 groups 400/817 loss=0.8320
Epoch 2 groups 500/817 loss=0.8319
Epoch 2 groups 600/817 loss=0.8320
Epoch 2 groups 700/817 loss=0.8320
Epoch 2 groups 8

## 7. Restore best student, tune Dev fusion, infer Test and create ZIP

In [7]:
best_path=OUTPUT_DIR/"best_student_kd.pt"
if not best_path.exists(): raise FileNotFoundError("No best_student_kd.pt was saved.")
best_obj=torch.load(best_path,map_location="cpu",weights_only=False)
student.load_state_dict(best_obj["student"],strict=True)
student_head.score.load_state_dict(best_obj["head"],strict=True)
student.eval();student_head.eval()
dev_cache=evaluate_student(dev_queries,dev_candidates,bm25_dev,"Final Dev")
best_alpha=0.;best_dev_metrics=metrics(dev_queries,predictions_from_cache(dev_cache,0.))
for a in ALPHA_GRID:
    m=metrics(dev_queries,predictions_from_cache(dev_cache,a))
    if m["micro_f1"]>best_dev_metrics["micro_f1"]:best_alpha,best_dev_metrics=a,m
bm25_dev_pred={q:[c for c,_ in bm25_dev.search(rec,top_n=7,exclude_id=q)] for q,rec in dev_queries.items()}
bm25_dev_m=metrics(dev_queries,bm25_dev_pred)
use_bm25=bm25_dev_m["micro_f1"]>best_dev_metrics["micro_f1"]
if use_bm25: print("Dev selected BM25-only fallback.")
print("Final Dev:",best_dev_metrics,"alpha:",best_alpha,"BM25:",bm25_dev_m)

bm25_test=BM25Index(test_candidates)
test_cache=evaluate_student(test_queries,test_candidates,bm25_test,"Test")
if use_bm25:
    test_predictions={q:[c for c,_ in bm25_test.search(rec,top_n=7,exclude_id=q)] for q,rec in test_queries.items()}
    selected="BM25-only"
else:
    test_predictions=predictions_from_cache(test_cache,best_alpha)
    selected="Model 3 KD cross-encoder + BM25 fusion"
assert set(test_predictions)==set(test_queries)
for q,ids in test_predictions.items():
    assert len(ids)==7 and len(set(ids))==7 and all(c in test_candidates for c in ids),(q,ids)
pred_path=OUTPUT_DIR/"prediction.json"
with open(pred_path,"w",encoding="utf-8") as f:json.dump(test_predictions,f,ensure_ascii=False,indent=2)
zip_path=OUTPUT_DIR/"phase7_model1_to_model3_kd_submission.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:z.write(pred_path,arcname="prediction.json")
with zipfile.ZipFile(zip_path) as z:assert z.namelist()==["prediction.json"]
summary={"experiment":"Model 1 teacher -> Model 3 student KD","teacher_checkpoint":M1_CKPT,
"student_checkpoint":M3_CKPT,"best_epoch":best_epoch,"best_dev_f1":best_dev,
"final_dev_metrics":best_dev_metrics,"bm25_dev_metrics":bm25_dev_m,"selected_method":selected,
"test_queries":len(test_predictions),"submission_zip":str(zip_path),"finished_at":stamp()}
with open(OUTPUT_DIR/"experiment_summary.json","w") as f:json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))

Final Dev: 25/118
Final Dev: 50/118
Final Dev: 75/118
Final Dev: 100/118
Final Dev: 118/118
Final Dev: {'micro_f1': 0.33463035019455256, 'micro_precision': 0.3644067796610169, 'micro_recall': 0.30935251798561153, 'tp': 301, 'fp': 525, 'fn': 672} alpha: 2.0 BM25: {'micro_f1': 0.330183435241801, 'micro_precision': 0.3595641646489104, 'micro_recall': 0.3052415210688592, 'tp': 297, 'fp': 529, 'fn': 676}
Test: 25/237
Test: 50/237
Test: 75/237
Test: 100/237
Test: 125/237
Test: 150/237
Test: 175/237
Test: 200/237
Test: 225/237
Test: 237/237
{
  "experiment": "Model 1 teacher -> Model 3 student KD",
  "teacher_checkpoint": "/kaggle/input/datasets/aaryant23/phase2-model3-ckpts/best_dev_f1.pt",
  "student_checkpoint": "/kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt",
  "best_epoch": 2,
  "best_dev_f1": 0.33463035019455256,
  "final_dev_metrics": {
    "micro_f1": 0.33463035019455256,
    "micro_precision": 0.3644067796610169,
    "micro_recall": 0.30935251798561153,

## Run notes and safeguards

1. Attach the **Model 1 Experiment 5 output dataset** containing `best_dev_f1.pt` and its matching Model 1 tokenizer.
2. Attach `phase2_model3_ckpts` containing `encdec_full_best*.pt` and `tokenizer (1).json`. The extra `.pt` file is ignored by filename-based discovery.
3. Attach the Phase-7 data repository access / Kaggle HF secret as in earlier experiments.
4. Select a Kaggle GPU (T4 recommended), then Run All.
5. Check the printed checkpoint paths before proceeding. If multiple teacher tokenizers/checkpoints exist, the notebook stops rather than silently choosing an unrelated one.
6. This notebook is syntax-checked, not executed on Kaggle. The first run must verify that the teacher checkpoint has `model` and `scorer` entries and that the Model 3 Phase-1 state dict matches the architecture strictly.